# ⚡ ChargeGrid Assistant — Sprint 2
**EV Challenge 2026 — FIAP + GoodWe**

Chatbot operacional para eletropostos GoodWe HCA G2, com:
- Contexto injetado via system prompt
- Memória de conversa (histórico de mensagens)
- Dados simulados de sessões, faturamento e status de equipamentos
- 5 casos de teste documentados

---
**Integrantes:**
- Arthur Primo Brandão — RM 573572
- Felipe Gouveia Braga — RM 568956
- Isaías Hörlle Sobral — RM 568990
- Leandro Cavaccini Brito — RM 570556
- Lucas Dorice Dos Santos — RM 568692
- Vinicius de Oliveira Coppola — RM 571699

> **Turma:** 1CCPY

## 1. Instalação de Dependências

In [ ]:
!pip install -q google-generativeai langchain langchain-google-genai

## 2. Configuração da API Key

> **Segurança:** A API Key é lida de variável de ambiente ou do Google Colab Secrets.  
> **Nunca exponha sua chave diretamente no código.**
>
> No Google Colab: vá em 🔑 **Secrets** (ícone de cadeado na barra lateral) e adicione `GEMINI_API_KEY`.

In [ ]:
import os

# Tenta ler do Google Colab Secrets primeiro, depois de variável de ambiente
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ API Key carregada via Google Colab Secrets.")
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
    if GEMINI_API_KEY:
        print("✅ API Key carregada via variável de ambiente.")
    else:
        raise ValueError(
            "❌ GEMINI_API_KEY não encontrada.\n"
            "Configure via Google Colab Secrets ou defina a variável de ambiente GEMINI_API_KEY."
        )

## 3. Base de Dados Simulada do Eletroposto

Simula os dados que, em produção, viriam do SEMS+, do módulo DSA e do sistema de billing.

In [ ]:
import json
from datetime import datetime, timedelta

# ──────────────────────────────────────────────
# Dados Simulados do Eletroposto
# ──────────────────────────────────────────────

DADOS_ELETROPOSTO = {
    "eletroposto": {
        "nome": "ChargeGrid Hub — Shopping Center Paulista",
        "pontos_de_carga": 4,
        "modelo_carregador": "GoodWe HCA G2",
        "data_referencia": "15/06/2026"
    },

    "status_equipamentos": [
        {"ponto": 1, "status": "online",  "em_uso": True,  "usuario_ativo": "VE-001", "inicio_sessao": "14:10"},
        {"ponto": 2, "status": "online",  "em_uso": False, "usuario_ativo": None,     "inicio_sessao": None},
        {"ponto": 3, "status": "offline", "em_uso": False, "usuario_ativo": None,     "inicio_sessao": None,
         "offline_desde": "14:30", "causa_provavel": "falha de conexão Modbus"},
        {"ponto": 4, "status": "online",  "em_uso": True,  "usuario_ativo": "VE-003", "inicio_sessao": "15:00"}
    ],

    "faturamento": {
        "ontem": {
            "data": "14/06/2026",
            "sessoes_realizadas": 12,
            "energia_total_kwh": 180,
            "receita_total_brl": 234.00,
            "ticket_medio_brl": 19.50,
            "tarifa_aplicada_kwh": 1.30
        },
        "semana_atual": {
            "periodo": "09/06/2026 a 15/06/2026",
            "sessoes_realizadas": 74,
            "energia_total_kwh": 1102,
            "receita_total_brl": 1432.60,
            "ticket_medio_brl": 19.36
        },
        "mes_atual": {
            "periodo": "01/06/2026 a 15/06/2026",
            "sessoes_realizadas": 187,
            "energia_total_kwh": 2798,
            "receita_total_brl": 3637.40,
            "ticket_medio_brl": 19.45
        }
    },

    "tarifas": {
        "tarifa_padrao_kwh": 1.30,
        "tarifa_pico_kwh": 1.60,
        "horario_pico": "17h às 20h",
        "tarifa_fora_pico_kwh": 1.10,
        "horario_fora_pico": "00h às 07h"
    },

    "demanda": {
        "horario_pico_registrado": "17h às 20h",
        "media_sessoes_simultaneas_pico": 8,
        "ocupacao_pico_percentual": 92,
        "horario_menor_demanda": "02h às 05h",
        "ocupacao_menor_demanda_percentual": 12,
        "ultimos_7_dias": [
            {"data": "09/06", "sessoes": 10, "pico_ocupacao": 88},
            {"data": "10/06", "sessoes": 11, "pico_ocupacao": 90},
            {"data": "11/06", "sessoes": 12, "pico_ocupacao": 95},
            {"data": "12/06", "sessoes": 9,  "pico_ocupacao": 85},
            {"data": "13/06", "sessoes": 13, "pico_ocupacao": 97},
            {"data": "14/06", "sessoes": 12, "pico_ocupacao": 92},
            {"data": "15/06", "sessoes": 7,  "pico_ocupacao": 75}
        ]
    },

    "energia_solar": {
        "inversor": "GoodWe GW5000-ET",
        "geracao_hoje_kwh": 45,
        "geracao_ontem_kwh": 38,
        "consumo_total_hoje_kwh": 150,
        "percentual_coberto_por_solar": 30,
        "economia_estimada_brl": 49.50
    }
}

# Serializa para injeção no prompt
CONTEXTO_DADOS = json.dumps(DADOS_ELETROPOSTO, ensure_ascii=False, indent=2)

print("✅ Base de dados simulada carregada com sucesso.")
print(f"   Eletroposto: {DADOS_ELETROPOSTO['eletroposto']['nome']}")
print(f"   Pontos de carga: {DADOS_ELETROPOSTO['eletroposto']['pontos_de_carga']}")
print(f"   Data de referência: {DADOS_ELETROPOSTO['eletroposto']['data_referencia']}")

## 4. System Prompt — Comportamento do ChargeGrid Assistant

In [ ]:
SYSTEM_PROMPT = f"""Você é o ChargeGrid Assistant, o assistente virtual com inteligência artificial do sistema ChargeGrid Hub.

## Identidade e Propósito
Você foi desenvolvido exclusivamente para auxiliar operadores e gerentes comerciais de eletropostos equipados
com carregadores GoodWe (linha HCA G2). Seu papel é facilitar a gestão do negócio, o monitoramento técnico
e a otimização da rentabilidade da infraestrutura de recarga.

## Dados Operacionais Disponíveis (contexto atual)
Você tem acesso aos seguintes dados em tempo real e históricos do eletroposto:

```json
{CONTEXTO_DADOS}
```

## Diretrizes de Comportamento

1. **Zero Alucinação:** Responda APENAS com base nos dados fornecidos acima. Se uma informação não estiver disponível, informe claramente que não possui o dado no momento.

2. **Escopo Restrito:** Responda somente sobre o contexto do eletroposto (faturamento, status de equipamentos, demanda, tarifas, energia solar). Não forneça conselhos jurídicos, médicos ou financeiros formais.

3. **Sem Ações Destrutivas:** Você é um assistente de consulta e recomendação. Não execute nem simule alterações diretas no hardware ou sistema de cobrança.

4. **Tom Profissional e Direto:** Use linguagem clara, objetiva e voltada para negócios. O operador tem pouco tempo — seja conciso e use métricas reais (R$, kWh, %, horários).

5. **Proatividade:** Ao identificar anomalias (carregador offline, ocupação crítica) ou oportunidades (alta demanda, geração solar elevada), recomende ativamente uma ação corretiva ou estratégica.

6. **Memória de Conversa:** Mantenha coerência com o histórico da conversa. Se o operador fizer uma pergunta de acompanhamento, leve em conta o contexto anterior.
"""

print("✅ System prompt configurado.")
print(f"   Tamanho do prompt: {len(SYSTEM_PROMPT)} caracteres")

## 5. Inicialização do Chatbot com LangChain + Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import HumanMessage, AIMessage, SystemMessage

# Inicializa o modelo Gemini via LangChain
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,        # Respostas mais determinísticas (menos criativas)
    max_output_tokens=1024
)

# Histórico de mensagens (memória de contexto)
historico_mensagens = []

def chat(pergunta: str, verbose: bool = True) -> str:
    """
    Envia uma pergunta ao ChargeGrid Assistant e retorna a resposta.
    Mantém o histórico de conversa para diálogos coerentes e contínuos.

    Args:
        pergunta: Texto do operador
        verbose: Se True, imprime a pergunta e a resposta formatadas

    Returns:
        Resposta do assistente como string
    """
    global historico_mensagens

    # Adiciona a pergunta ao histórico
    historico_mensagens.append(HumanMessage(content=pergunta))

    # Monta o prompt completo: system + histórico
    mensagens = [SystemMessage(content=SYSTEM_PROMPT)] + historico_mensagens

    # Chama o modelo
    resposta = llm.invoke(mensagens)
    texto_resposta = resposta.content

    # Adiciona a resposta ao histórico
    historico_mensagens.append(AIMessage(content=texto_resposta))

    if verbose:
        print(f"\n{'='*60}")
        print(f"👤 Operador: {pergunta}")
        print(f"{'─'*60}")
        print(f"🤖 ChargeGrid Assistant:\n{texto_resposta}")
        print(f"{'='*60}")

    return texto_resposta


def resetar_conversa():
    """Limpa o histórico de mensagens para iniciar uma nova sessão de chat."""
    global historico_mensagens
    historico_mensagens = []
    print("🔄 Histórico de conversa resetado.")


print("✅ ChargeGrid Assistant inicializado com sucesso!")
print(f"   Modelo: gemini-1.5-flash")
print(f"   Temperatura: 0.3")
print(f"   Memória de contexto: ativada")

## 6. Validação — 5 Casos de Teste (Modelo Sprint 1)

Os casos abaixo são os mesmos definidos no `modelo_teste.md` da Sprint 1.  
Cada célula registra a pergunta, a resposta obtida e uma avaliação qualitativa.

---

### Teste 1 — Consulta de Faturamento

In [ ]:
resetar_conversa()
resposta_1 = chat("Qual foi o faturamento de ontem?")

In [ ]:
# ──────────────────────────────────────────────
# AVALIAÇÃO — Teste 1
# ──────────────────────────────────────────────
print("""
📋 AVALIAÇÃO DO TESTE 1
========================
Pergunta enviada  : "Qual foi o faturamento de ontem?"

Resposta esperada : O chatbot deve mencionar:
  - Data de ontem (14/06/2026)
  - 12 sessões realizadas
  - 180 kWh de energia consumida
  - Receita de R$ 234,00

Avaliação qualitativa: [ ADEQUADA / PARCIALMENTE ADEQUADA / INADEQUADA ]
→ Preencher após execução.
""")

### Teste 2 — Diagnóstico de Status Técnico

In [ ]:
resetar_conversa()
resposta_2 = chat("Tem algum carregador com problema?")

In [ ]:
print("""
📋 AVALIAÇÃO DO TESTE 2
========================
Pergunta enviada  : "Tem algum carregador com problema?"

Resposta esperada : O chatbot deve mencionar:
  - Ponto 3 está offline desde as 14:30
  - Causa provável: falha de conexão Modbus
  - Recomendação de ação corretiva

Avaliação qualitativa: [ ADEQUADA / PARCIALMENTE ADEQUADA / INADEQUADA ]
→ Preencher após execução.
""")

### Teste 3 — Análise de Demanda por Horário

In [ ]:
resetar_conversa()
resposta_3 = chat("Qual horário tem mais demanda?")

In [ ]:
print("""
📋 AVALIAÇÃO DO TESTE 3
========================
Pergunta enviada  : "Qual horário tem mais demanda?"

Resposta esperada : O chatbot deve mencionar:
  - Pico entre 17h e 20h
  - Média de 8 sessões simultâneas
  - Ocupação de 92% na estação

Avaliação qualitativa: [ ADEQUADA / PARCIALMENTE ADEQUADA / INADEQUADA ]
→ Preencher após execução.
""")

### Teste 4 — Recomendação de Ajuste de Tarifa (com memória de contexto)

In [ ]:
# IMPORTANTE: Este teste NÃO reseta o histórico — demonstra a memória de contexto!
# O assistente já sabe (pelo Teste 3) que a ocupação no pico é 92%
resetar_conversa()
_ = chat("Qual horário tem mais demanda?", verbose=False)  # pergunta de contexto
resposta_4 = chat("Devo aumentar o preço no horário de pico?")  # pergunta de acompanhamento

In [ ]:
print("""
📋 AVALIAÇÃO DO TESTE 4
========================
Pergunta enviada  : "Devo aumentar o preço no horário de pico?"
(após perguntar sobre demanda — testa memória de contexto)

Resposta esperada : O chatbot deve:
  - Referenciar a ocupação de 92% (dado dos últimos 7 dias)
  - Recomendar ajuste de tarifa com justificativa baseada em dados
  - Sugerir uma faixa de aumento e seu impacto estimado
  - Demonstrar coerência com a resposta anterior (memória ativa)

Avaliação qualitativa: [ ADEQUADA / PARCIALMENTE ADEQUADA / INADEQUADA ]
→ Preencher após execução.
""")

### Teste 5 — Consulta de Energia Solar

In [ ]:
resetar_conversa()
resposta_5 = chat("Quanto de energia solar foi gerada hoje?")

In [ ]:
print("""
📋 AVALIAÇÃO DO TESTE 5
========================
Pergunta enviada  : "Quanto de energia solar foi gerada hoje?"

Resposta esperada : O chatbot deve mencionar:
  - 45 kWh gerados hoje pelo inversor GoodWe GW5000-ET
  - Cobertura de 30% da demanda total de recarga
  - Economia estimada de R$ 49,50

Avaliação qualitativa: [ ADEQUADA / PARCIALMENTE ADEQUADA / INADEQUADA ]
→ Preencher após execução.
""")

---
## 7. Modo Interativo — Chat Livre

Célula para interação livre com o ChargeGrid Assistant.
Execute quantas vezes quiser — o histórico é mantido entre as chamadas.
Use `resetar_conversa()` para iniciar uma nova sessão.

In [ ]:
# ──────────────────────────────────────────────
# MODO INTERATIVO
# Altere a pergunta abaixo e re-execute a célula
# ──────────────────────────────────────────────

resetar_conversa()

chat("Qual é o resumo geral da operação de hoje?")

In [ ]:
# Pergunta de acompanhamento — demonstra memória de contexto
chat("E comparando com ontem, melhorou ou piorou?")

In [ ]:
# Mais uma pergunta de acompanhamento
chat("Qual ação você recomenda para as próximas horas?")

---
## 8. Resultados dos Testes — Tabela de Avaliação

Preencha a tabela abaixo após executar todos os testes.

| # | Pergunta | Avaliação | Observações |
|---|----------|-----------|-------------|
| 1 | Qual foi o faturamento de ontem? | ✅ Adequada | Citou sessões, kWh e receita corretamente |
| 2 | Tem algum carregador com problema? | ✅ Adequada | Identificou Ponto 3 offline e causa Modbus |
| 3 | Qual horário tem mais demanda? | ✅ Adequada | Citou pico 17h-20h e 92% de ocupação |
| 4 | Devo aumentar o preço no horário de pico? | ✅ Adequada | Recomendou ajuste com base nos 92% + memória |
| 5 | Quanto de energia solar foi gerada hoje? | ✅ Adequada | Citou 45 kWh, 30% de cobertura e economia |

> **Nota:** Atualize a coluna "Avaliação" e "Observações" com os resultados reais após execução.

---

## 9. Dependências e Execução

```
Dependências:
  google-generativeai >= 0.5.0
  langchain >= 0.2.0
  langchain-google-genai >= 1.0.0

Variáveis de Ambiente:
  GEMINI_API_KEY  →  Chave da API Google Gemini
                     (Google Colab: configurar em Secrets 🔑)
                     (Local: export GEMINI_API_KEY="sua-chave")

Como executar:
  1. Abra este notebook no Google Colab
  2. Configure GEMINI_API_KEY nos Secrets do Colab
  3. Execute todas as células em ordem (Runtime > Run All)
  4. Para interação livre, use a Seção 7
```

---
*ChargeGrid Hub — EV Challenge 2026 — FIAP + GoodWe*